In [ ]:
# Summary Dashboard
print("\n" + "="*60)
print("📊 PROJECT SUMMARY & KEY INSIGHTS")
print("="*60)

print(f"""
✅ DATA PROCESSING:
  • Total Records: {len(df)}
  • Categories: {df['category'].nunique()}
  • Locations: {df['location'].nunique()}
  • Priority Levels: 3 (Low, Medium, High)

📈 KEY STATISTICS:
  • Average Complaint Length: {df['text_length'].mean():.0f} characters
  • Average Words per Complaint: {df['word_count'].mean():.0f}
  • Average Sentiment Polarity: {df['sentiment_polarity'].mean():.3f}
  
🏆 BEST MODELS:
  • Classification: {best_clf_name} (F1: {clf_results[best_clf_name]['F1-Score']:.4f})
  • Priority Prediction: {best_priority_name} (F1: {priority_results[best_priority_name]['F1-Score']:.4f})

🔍 INSIGHTS:
  • Most Common Category: {df['category'].value_counts().index[0]} ({df['category'].value_counts().values[0]} cases)
  • Most Common Priority: {df['priority'].value_counts().index[0]} ({df['priority'].value_counts().values[0]} cases)
  • Top Location: {df['location'].value_counts().index[0]} ({df['location'].value_counts().values[0]} cases)
  • Peak Complaint Hour: {df['hour'].mode()[0]}:00

✨ FEATURES ENGINEERED:
  • Text-based: Length, word count, sentiment (polarity & subjectivity)
  • Time-based: Hour, day of week, month, is_weekend
  • Location-based: Complaint count, density, top location flag
  • Categorical: Category, department encoded features

🎯 RECOMMENDATIONS:
  1. Use {best_clf_name} for category classification
  2. Use {best_priority_name} for priority prediction
  3. Focus on sentiment analysis for priority weighting
  4. Monitor {df['location'].value_counts().index[0]} for patterns
  5. Investigate peak hours ({df['hour'].mode()[0]}:00) for resource allocation
""")

print("="*60)
print("✅ ANALYSIS COMPLETE")
print("="*60)

In [ ]:
# 10. PREDICTION ON NEW GRIEVANCES
print("\n🔮 PREDICTION ON NEW GRIEVANCES")
print("="*50)

# Create a few new grievance examples
new_complaints = [
    "Road is completely damaged with huge potholes making it dangerous",
    "Street lights have not been working for 2 weeks affecting safety",
    "Minor issue with parking space",
]

print("\nProcessing new complaints...")

for i, complaint in enumerate(new_complaints):
    print(f"\n--- Complaint {i+1} ---")
    print(f"Text: {complaint}")
    
    # Process the complaint
    processor = TextProcessor()
    cleaned = processor.clean_text(complaint)
    sentiment = processor.get_sentiment(complaint)
    stats = processor.get_text_statistics(complaint)
    
    # Create feature vector (this is a simplified version)
    print(f"\nFeatures:")
    print(f"  Text Length: {stats['text_length']}")
    print(f"  Word Count: {stats['word_count']}")
    print(f"  Sentence Count: {stats['sentence_count']}")
    print(f"  Sentiment Polarity: {sentiment['polarity']:.3f}")
    print(f"  Sentiment Subjectivity: {sentiment['subjectivity']:.3f}")
    
    print(f"\nAdvice:")
    if sentiment['polarity'] < -0.3:
        print(f"  ⚠️ Very negative tone - Higher priority recommended")
    elif sentiment['polarity'] < 0:
        print(f"  ⚠️ Negative tone - Standard priority review")
    else:
        print(f"  ✓ Neutral/positive tone - Can be routine")

## Section 10: Prediction on New Grievances

In [ ]:
# Confusion Matrices Visualization
import plotly.figure_factory as ff

categories = [list(category_mapping.keys())[i] for i in sorted(category_mapping.values())]

# Classification confusion matrix
fig = ff.create_annotated_heatmap(
    z=cm_clf,
    x=categories,
    y=categories,
    colorscale='Blues',
    showscale=True
)
fig.update_layout(
    title='Confusion Matrix - Category Classification',
    xaxis_title='Predicted',
    yaxis_title='Actual',
    height=600
)
fig.show()

# Priority confusion matrix
priority_labels = ['Low', 'Medium', 'High']
fig = ff.create_annotated_heatmap(
    z=cm_priority,
    x=priority_labels,
    y=priority_labels,
    colorscale='Reds',
    showscale=True
)
fig.update_layout(
    title='Confusion Matrix - Priority Prediction',
    xaxis_title='Predicted',
    yaxis_title='Actual',
    height=600
)
fig.show()

In [ ]:
# 9. VISUALIZATION DASHBOARD
print("\n📈 CREATING VISUALIZATION DASHBOARD")
print("="*50)

# Feature importance from Random Forest
if 'feature_importances_' in dir(best_clf):
    importances = best_clf.feature_importances_
    feature_names = X_encoded.columns
    
    # Get top 15 features
    top_indices = np.argsort(importances)[-15:]
    
    fig = px.barh(
        y=[feature_names[i] for i in top_indices],
        x=[importances[i] for i in top_indices],
        title=f'Top 15 Feature Importances - {best_clf_name}',
        labels={'x': 'Importance', 'y': 'Feature'},
        color=[importances[i] for i in top_indices],
        color_continuous_scale='Viridis'
    )
    fig.update_layout(height=600, showlegend=False)
    fig.show()
    print("✓ Feature importance plot created")

## Section 9: Visualization Dashboard and Insights

In [ ]:
# Compare all models
fig = go.Figure()

# Add classification models
fig.add_trace(go.Bar(
    name='Classification Models',
    x=list(clf_results.keys()),
    y=[clf_results[m]['F1-Score'] for m in clf_results],
    marker_color='lightblue'
))

# Add priority prediction models  
fig.add_trace(go.Bar(
    name='Priority Models',
    x=list(priority_results.keys()),
    y=[priority_results[m]['F1-Score'] for m in priority_results],
    marker_color='lightcoral'
))

fig.update_layout(
    title='Model Comparison: F1-Scores',
    yaxis_title='F1-Score',
    barmode='group',
    height=500
)
fig.show()

In [ ]:
# 8. MODEL EVALUATION
print("\n📊 MODEL EVALUATION & COMPARISON")
print("="*50)

from sklearn.metrics import confusion_matrix, classification_report

# Get best models
best_clf_name = clf_df['F1-Score'].idxmax()
best_clf = clf_models[best_clf_name]
y_pred_clf = best_clf.predict(X_test)

best_priority_name = priority_df['F1-Score'].idxmax()
best_priority = priority_trained_models[best_priority_name]
y_pred_priority = best_priority.predict(X_test_p)

print(f"\n🏆 Best Classification Model: {best_clf_name}")
print(f"   Accuracy: {clf_results[best_clf_name]['Accuracy']:.4f}")
print(f"   F1-Score: {clf_results[best_clf_name]['F1-Score']:.4f}")

print(f"\n🏆 Best Priority Prediction Model: {best_priority_name}")
print(f"   Accuracy: {priority_results[best_priority_name]['Accuracy']:.4f}")
print(f"   F1-Score: {priority_results[best_priority_name]['F1-Score']:.4f}")

# Confusion Matrices
print(f"\n🔲 Classification Confusion Matrix:")
cm_clf = confusion_matrix(y_test, y_pred_clf)
print(cm_clf)

print(f"\n🔲 Priority Prediction Confusion Matrix:")
cm_priority = confusion_matrix(y_test_p, y_pred_priority)
print(cm_priority)

## Section 8: Model Evaluation and Comparison

In [ ]:
# 7. PRIORITY PREDICTION MODELS
print("\n⚠️  BUILDING PRIORITY PREDICTION MODELS")
print("="*50)

# Target: Priority
y_priority = df['priority']
priority_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
y_priority_encoded = y_priority.map(priority_mapping)

# Split data
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_encoded, y_priority_encoded, test_size=0.2, random_state=42, stratify=y_priority_encoded
)

priority_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, verbosity=0)
}

priority_results = {}
priority_trained_models = {}

for name, model in priority_models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train_p, y_train_p)
    priority_trained_models[name] = model
    
    y_pred_p = model.predict(X_test_p)
    
    accuracy = accuracy_score(y_test_p, y_pred_p)
    precision = precision_score(y_test_p, y_pred_p, average='weighted', zero_division=0)
    recall = recall_score(y_test_p, y_pred_p, average='weighted', zero_division=0)
    f1 = f1_score(y_test_p, y_pred_p, average='weighted', zero_division=0)
    
    priority_results[name] = {'Accuracy': accuracy, 'Precision': precision, 'Recall': recall, 'F1-Score': f1}
    print(f"  ✓ Accuracy: {accuracy:.4f} | F1: {f1:.4f}")

# Display results
priority_df = pd.DataFrame(priority_results).T
print(f"\n{'='*50}")
print("PRIORITY PREDICTION MODEL RESULTS:")
print(f"{'='*50}")
print(priority_df)

## Section 7: Model Building for Priority Prediction

In [ ]:
# 6. MODEL BUILDING FOR CLASSIFICATION
print("🚀 BUILDING CLASSIFICATION MODELS")
print("="*50)

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Prepare features
drop_cols = ['complaint_id', 'category', 'priority', 'location', 'complaint_text', 
             'complaint_text_cleaned', 'timestamp', 'date', 'day_of_week', 'month']
X = df.drop(columns=drop_cols, errors='ignore')

# Encode categorical columns
X_encoded = pd.get_dummies(X)

# Target: Category
y_category = df['category']
category_mapping = {cat: i for i, cat in enumerate(sorted(y_category.unique()))}
y_category_encoded = y_category.map(category_mapping)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_category_encoded, test_size=0.2, random_state=42, stratify=y_category_encoded
)

print(f"\nTraining set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"Features: {X_encoded.shape[1]}")

# Train models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Naive Bayes': GaussianNB(),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, verbosity=0)
}

clf_results = {}
clf_models = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    clf_models[name] = model
    
    y_pred = model.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    clf_results[name] = {'Accuracy': accuracy, 'Precision': precision, 'Recall': recall, 'F1-Score': f1}
    print(f"  ✓ Accuracy: {accuracy:.4f} | F1: {f1:.4f}")

# Display results
clf_df = pd.DataFrame(clf_results).T
print(f"\n{'='*50}")
print("CLASSIFICATION MODEL RESULTS:")
print(f"{'='*50}")
print(clf_df)

## Section 6: Model Building for Classification

In [ ]:
# Visualize sentiment distribution
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Sentiment Polarity Distribution", "Sentiment Subjectivity Distribution")
)

fig.add_trace(
    go.Histogram(x=df['sentiment_polarity'], name='Polarity', nbinsx=30),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(x=df['sentiment_subjectivity'], name='Subjectivity', nbinsx=30),
    row=1, col=2
)

fig.update_layout(height=400, showlegend=False, title_text="Sentiment Analysis Distribution")
fig.show()

# Sentiment by priority
print("\nAverage Sentiment by Priority:")
sentiment_by_priority = df.groupby('priority')[['sentiment_polarity', 'sentiment_subjectivity']].mean()
print(sentiment_by_priority)

In [ ]:
# 5. FEATURE ENGINEERING
print("🔧 FEATURE ENGINEERING")
print("="*50)

from utils import FeatureEngineer, FeatureScaler
from sklearn.feature_extraction.text import TfidfVectorizer

feature_engineer = FeatureEngineer()

# Extract text statistics (already done above)
print("\n✓ Text statistics extracted")

# Extract time features
time_features = FeatureEngineer.extract_time_features(df['timestamp'])
print(f"✓ Time features: {time_features.shape[1]} features")

# Extract location features
location_features = FeatureEngineer.extract_location_features(df['location'])
print(f"✓ Location features: {location_features.shape[1]} features")

# Extract sentiment analysis
print("\nAnalyzing sentiment...")
sentiments = []
for text in df['complaint_text']:
    sentiment = text_processor.get_sentiment(text)
    sentiments.append(sentiment)

df['sentiment_polarity'] = [s['polarity'] for s in sentiments]
df['sentiment_subjectivity'] = [s['subjectivity'] for s in sentiments]
print(f"✓ Sentiment features extracted")

# Show sentiment statistics
print(f"\nSentiment Polarity Statistics:")
print(df['sentiment_polarity'].describe())

## Section 5: Feature Engineering

In [ ]:
# 4E. CATEGORY-PRIORITY CROSS ANALYSIS
print("\n🔀 CATEGORY-PRIORITY RELATIONSHIP")
print("="*50)

# Create heatmap
category_priority = pd.crosstab(df['category'], df['priority'])
print(f"\nCategory-Priority Crosstab:")
print(category_priority)

fig = px.imshow(category_priority,
               labels=dict(x="Priority", y="Category", color="Count"),
               title="Heatmap: Complaint Category vs Priority Level",
               color_continuous_scale="YlOrRd",
               aspect="auto")
fig.update_layout(height=600)
fig.show()

In [ ]:
# 4D. TEMPORAL ANALYSIS
print("\n⏰ TEMPORAL ANALYSIS")
print("="*50)

# Convert timestamp to datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Extract time features
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.day_name()
df['month'] = df['timestamp'].dt.month_name()
df['date'] = df['timestamp'].dt.date

# Complaints by hour of day
hourly_complaints = df['hour'].value_counts().sort_index()
fig = px.bar(x=hourly_complaints.index, y=hourly_complaints.values,
            labels={'x': 'Hour of Day', 'y': 'Number of Complaints'},
            title='Complaints Distribution by Hour')
fig.show()

# Complaints by day of week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_complaints = df['day_of_week'].value_counts().reindex(day_order)
fig = px.bar(x=daily_complaints.index, y=daily_complaints.values,
            labels={'x': 'Day of Week', 'y': 'Number of Complaints'},
            title='Complaints Distribution by Day of Week',
            color=daily_complaints.values,
            color_continuous_scale='Blues')
fig.show()

In [ ]:
# 4C. LOCATION-BASED ANALYSIS
print("\n📍 LOCATION-BASED ANALYSIS")
print("="*50)

location_counts = df['location'].value_counts()
print(f"\nComplaints by Location (Top 10):")
print(location_counts.head(10))

# Horizontal bar chart
fig = px.barh(y=location_counts.head(10).index[::-1], 
             x=location_counts.head(10).values[::-1],
             labels={'x': 'Number of Grievances', 'y': 'Location'},
             title='Top 10 Locations by Grievance Count',
             color=location_counts.head(10).values[::-1],
             color_continuous_scale='Spectral')
fig.update_layout(height=500, showlegend=False)
fig.show()

In [ ]:
# 4B. PRIORITY DISTRIBUTION
print("\n⚠️ PRIORITY DISTRIBUTION ANALYSIS")
print("="*50)

priority_counts = df['priority'].value_counts()
priority_order = {'Low': 0, 'Medium': 1, 'High': 2}
priority_counts = priority_counts.reindex(['Low', 'Medium', 'High'])

print(f"\nComplaints by Priority:")
print(priority_counts)

# Pie chart
colors = {'High': '#d62728', 'Medium': '#ff7f0e', 'Low': '#2ca02c'}
fig = px.pie(values=priority_counts.values, names=priority_counts.index,
            title='Grievance Distribution by Priority Level',
            color=priority_counts.index,
            color_discrete_map=colors)
fig.show()

print(f"\nPriority Percentages:")
print(priority_counts / len(df) * 100)

In [ ]:
# 4A. CATEGORY DISTRIBUTION
print("📊 COMPLAINT DISTRIBUTION ANALYSIS")
print("="*50)

category_counts = df['category'].value_counts()
print(f"\nComplaints by Category:")
print(category_counts)

# Visualize category distribution
fig = px.bar(x=category_counts.index, y=category_counts.values,
            labels={'x': 'Complaint Category', 'y': 'Count'},
            title='Grievance Distribution by Category',
            color=category_counts.values,
            color_continuous_scale='Viridis')
fig.update_layout(height=500, showlegend=False)
fig.show()

# Show percentages
print(f"\nCategory Percentages:")
print(category_counts / len(df) * 100)

## Section 4: Exploratory Data Analysis (EDA)

In [ ]:
# Preprocess data
import sys
sys.path.insert(0, '..')

from utils import TextProcessor

# Initialize text processor
text_processor = TextProcessor()

# Clean text
print("🧹 Cleaning complaint text...")
df['complaint_text_cleaned'] = df['complaint_text'].apply(text_processor.clean_text)

# Extract text statistics
print("📊 Extracting text statistics...")
text_stats = []
for text in df['complaint_text']:
    stats = text_processor.get_text_statistics(text)
    text_stats.append(stats)

text_stats_df = pd.DataFrame(text_stats)
df = pd.concat([df, text_stats_df], axis=1)

print("✓ Text preprocessing complete!")
print(f"\nText Statistics Summary:")
print(text_stats_df.describe())

## Section 3: Data Preprocessing and Text Cleaning

In [ ]:
# Check for missing values
print("🔎 Missing Values:")
missing = df.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print("No missing values found!")

print(f"\n📍 Data Types:")
print(df.dtypes)

print(f"\n🏷️ Unique Values:")
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    print(f"  {col}: {df[col].nunique()} unique values")

## Section 2: Load and Explore Data

In [ ]:
# 2. LOAD AND EXPLORE GRIEVANCE DATA
# First, generate sample data if it doesn't exist
import os
if not os.path.exists('../data/grievances_dataset.csv'):
    print("Generating sample data...")
    exec(open('../scripts/generate_sample_data.py').read())

# Load the dataset
df = pd.read_csv('../data/grievances_dataset.csv')

print(f"📊 Dataset Shape: {df.shape}")
print(f"\n📋 Column Information:")
print(df.info())
print(f"\n📈 Basic Statistics:")
print(df.describe())
print(f"\n🔍 First 5 Records:")
df.head()

## Section 1: Import Required Libraries

In [ ]:
# 1. IMPORT REQUIRED LIBRARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All libraries imported successfully!")

# Public Grievance Classification & Priority Prediction
## Exploratory Data Analysis (EDA) & Machine Learning Pipeline

This notebook demonstrates a comprehensive ML-based system for analyzing, classifying, and predicting priority levels of public grievances.

**Key Objectives:**
- Analyze grievance data patterns
- Extract meaningful features
- Build and evaluate ML models
- Generate insights through visualization